# FramePack on Colab（無料 T4 向け）

ComfyUI + [ComfyUI-FramePackWrapper](https://github.com/kijai/ComfyUI-FramePackWrapper)（kijai 版）で、**画像1枚から動画を生成**します。

## 無料 T4 向けの構成
| 項目 | 設定 | 理由 |
|---|---|---|
| Transformer | `FramePackI2V_HY_fp8_e4m3fn`（約半分のサイズ） | 無料 Colab の RAM（約12GB）に収めるため |
| Text Encoder | `llava_llama3_fp8_scaled` + `clip_l` | RAM / ディスク節約 |
| load_device | `offload_device` | VRAM を使い切らないため |
| 初期設定 | 3秒 / 解像度バケット 512 / attention=sdpa | まず動作確認を優先 |

## 使い方
1. Step 1〜3 を順に実行（初回はモデルのダウンロードで時間がかかります）
2. Step 4 で ComfyUI を起動し、表示された URL を開く
3. 左上のワークフロー一覧から **`framepack_i2v`** を開く
4. **LoadImage（2か所）** に入力画像を指定（同じ画像で構いません）、プロンプトを入力して実行
5. 動画は Google Drive の `MyDrive/ComfyUI_Output` に保存されます

> ⚠️ 無料 T4 は生成にかなり時間がかかります。まずは3秒・低解像度で動作確認し、徐々に伸ばしてください。
> RAM 不足（セッションクラッシュ）が出たら、報告してください。設定を調整します。

## Step 1: 環境セットアップ
ComfyUI 本体と、FramePack に必要なカスタムノードを導入します。

In [ ]:
#@title Step 1: ComfyUI とカスタムノードの導入

import os
WORKSPACE = "/content/ComfyUI"
CUSTOM_DIR = f"{WORKSPACE}/custom_nodes"

![ ! -d {WORKSPACE} ] && git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
%cd {WORKSPACE}
!git pull

# torch は Colab 既存のものをそのまま使う
!pip install -q -r requirements.txt
!pip install -q accelerate "diffusers>=0.33.1" "transformers>=4.46.2" einops safetensors sentencepiece
!pip install -q huggingface_hub

# FramePack ラッパー + ワークフローで使うノード
NODES = {
    "ComfyUI-FramePackWrapper":  "https://github.com/kijai/ComfyUI-FramePackWrapper",
    "ComfyUI-KJNodes":           "https://github.com/kijai/ComfyUI-KJNodes",            # Set/Get ノード
    "ComfyUI_essentials":        "https://github.com/cubiq/ComfyUI_essentials",         # ImageResize+
    "ComfyUI-VideoHelperSuite":  "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",  # 動画出力
}
for name, url in NODES.items():
    path = f"{CUSTOM_DIR}/{name}"
    if not os.path.isdir(path):
        !git clone {url} {path}
    req = f"{path}/requirements.txt"
    if os.path.exists(req):
        !pip install -q -r {req}

print("\n✅ Step 1 完了")

## Step 2: 出力先の設定
生成した動画を Google Drive に保存します（セッション切れ対策）。

In [ ]:
#@title Step 2: 出力先を Google Drive に設定

import os, shutil
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

WORKSPACE = globals().get("WORKSPACE", "/content/ComfyUI")
drive_output = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(drive_output, exist_ok=True)

local_output = f"{WORKSPACE}/output"
if os.path.islink(local_output):
    os.unlink(local_output)
elif os.path.isdir(local_output):
    shutil.rmtree(local_output)
os.symlink(drive_output, local_output)

print("output ->", os.path.realpath(local_output))

## Step 3: モデルのダウンロード

| ファイル | 配置先 | サイズ目安 |
|---|---|---|
| `FramePackI2V_HY_fp8_e4m3fn.safetensors` | `models/diffusion_models` | 約13GB |
| `llava_llama3_fp8_scaled.safetensors` | `models/text_encoders` | 約9GB |
| `clip_l.safetensors` | `models/text_encoders` | 約0.2GB |
| `hunyuan_video_vae_bf16.safetensors` | `models/vae` | 約0.5GB |
| `sigclip_vision_patch14_384.safetensors` | `models/clip_vision` | 約0.9GB |

In [ ]:
#@title Step 3: FramePack モデルのダウンロード

from huggingface_hub import hf_hub_download
import os, shutil

WORKSPACE  = globals().get("WORKSPACE", "/content/ComfyUI")
MODEL_BASE = f"{WORKSPACE}/models"

# (repo, repo内パス, 配置先ディレクトリ)
FILES = [
    ("Kijai/HunyuanVideo_comfy", "FramePackI2V_HY_fp8_e4m3fn.safetensors", "diffusion_models"),
    ("Comfy-Org/HunyuanVideo_repackaged", "split_files/text_encoders/llava_llama3_fp8_scaled.safetensors", "text_encoders"),
    ("Comfy-Org/HunyuanVideo_repackaged", "split_files/text_encoders/clip_l.safetensors", "text_encoders"),
    ("Comfy-Org/HunyuanVideo_repackaged", "split_files/vae/hunyuan_video_vae_bf16.safetensors", "vae"),
    ("Comfy-Org/sigclip_vision_384", "sigclip_vision_patch14_384.safetensors", "clip_vision"),
]

for repo, path, sub in FILES:
    dst_dir = f"{MODEL_BASE}/{sub}"
    os.makedirs(dst_dir, exist_ok=True)
    dst = f"{dst_dir}/{os.path.basename(path)}"
    if os.path.exists(dst):
        print("⏭️ 既にあります:", os.path.basename(path))
        continue
    print("⬇️", repo, path)
    p = hf_hub_download(repo, path, local_dir=f"{WORKSPACE}/_hf_tmp")
    shutil.move(p, dst)
shutil.rmtree(f"{WORKSPACE}/_hf_tmp", ignore_errors=True)
print("\n✅ Step 3 完了")

## Step 3b: ワークフローの配置
FramePack の動画生成ワークフロー（kijai 版サンプルを T4 / fp8 向けに調整）を ComfyUI に登録します。

In [ ]:
#@title Step 3b: ワークフロー framepack_i2v を配置

import os, json

WORKSPACE = globals().get("WORKSPACE", "/content/ComfyUI")
WF_DIR = f"{WORKSPACE}/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

WORKFLOW = json.loads(r'''{"id":"ce2cb810-7775-4564-8928-dd5bed1053cd","revision":0,"last_node_id":69,"last_link_id":158,"nodes":[{"id":15,"type":"ConditioningZeroOut","pos":[1346.0872802734375,263.21856689453125],"size":[317.4000244140625,26],"flags":{"collapsed":true},"order":18,"mode":0,"inputs":[{"name":"conditioning","type":"CONDITIONING","link":118}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[108]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"ConditioningZeroOut"},"widgets_values":[],"color":"#332922","bgcolor":"#593930"},{"id":13,"type":"DualCLIPLoader","pos":[320.9956359863281,166.8336181640625],"size":[340.2243957519531,130],"flags":{},"order":0,"mode":0,"inputs":[],"outputs":[{"name":"CLIP","type":"CLIP","links":[102]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"DualCLIPLoader"},"widgets_values":["clip_l.safetensors","llava_llama3_fp8_scaled.safetensors","hunyuan_video","default"],"color":"#432","bgcolor":"#653"},{"id":54,"type":"DownloadAndLoadFramePackModel","pos":[1256.5235595703125,-277.76226806640625],"size":[315,130],"flags":{},"order":1,"mode":4,"inputs":[{"name":"compile_args","shape":7,"type":"FRAMEPACKCOMPILEARGS","link":null}],"outputs":[{"name":"model","type":"FramePackMODEL","links":null}],"properties":{"aux_id":"kijai/ComfyUI-FramePackWrapper","ver":"49fe507eca8246cc9d08a8093892f40c1180e88f","Node name for S&R":"DownloadAndLoadFramePackModel"},"widgets_values":["lllyasviel/FramePackI2V_HY","bf16","disabled","sdpa"]},{"id":55,"type":"MarkdownNote","pos":[567.05908203125,-628.8865966796875],"size":[459.8609619140625,285.9714660644531],"flags":{},"order":2,"mode":0,"inputs":[],"outputs":[],"properties":{},"widgets_values":["Model links:\n\n[https://huggingface.co/Kijai/HunyuanVideo_comfy/blob/main/FramePackI2V_HY_fp8_e4m3fn.safetensors](https://huggingface.co/Kijai/HunyuanVideo_comfy/blob/main/FramePackI2V_HY_fp8_e4m3fn.safetensors)\n\n[https://huggingface.co/Kijai/HunyuanVideo_comfy/blob/main/FramePackI2V_HY_bf16.safetensors](https://huggingface.co/Kijai/HunyuanVideo_comfy/blob/main/FramePackI2V_HY_bf16.safetensors)\n\nsigclip:\n\n[https://huggingface.co/Comfy-Org/sigclip_vision_384/tree/main](https://huggingface.co/Comfy-Org/sigclip_vision_384/tree/main)\n\ntext encoder and VAE:\n\n[https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/tree/main/split_files](https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/tree/main/split_files)"],"color":"#432","bgcolor":"#653"},{"id":17,"type":"CLIPVisionEncode","pos":[1545.9541015625,359.1331481933594],"size":[380.4000244140625,78],"flags":{},"order":23,"mode":0,"inputs":[{"name":"clip_vision","type":"CLIP_VISION","link":149},{"name":"image","type":"IMAGE","link":116}],"outputs":[{"name":"CLIP_VISION_OUTPUT","type":"CLIP_VISION_OUTPUT","links":[141]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"CLIPVisionEncode"},"widgets_values":["center"],"color":"#233","bgcolor":"#355"},{"id":64,"type":"GetNode","pos":[1554.2071533203125,486.79547119140625],"size":[210,60],"flags":{"collapsed":true},"order":3,"mode":0,"inputs":[],"outputs":[{"name":"CLIP_VISION","type":"CLIP_VISION","links":[149]}],"title":"Get_ClipVisionModle","properties":{},"widgets_values":["ClipVisionModle"],"color":"#233","bgcolor":"#355"},{"id":48,"type":"GetImageSizeAndCount","pos":[1259.2060546875,626.8657836914062],"size":[277.20001220703125,86],"flags":{},"order":21,"mode":0,"inputs":[{"name":"image","type":"IMAGE","link":125}],"outputs":[{"name":"image","type":"IMAGE","links":[116,156]},{"label":"704 width","name":"width","type":"INT","links":null},{"label":"544 height","name":"height","type":"INT","links":null},{"label":"1 count","name":"count","type":"INT","links":null}],"properties":{"cnr_id":"comfyui-kjnodes","ver":"8ecf5cd05e0a1012087b0da90eea9a13674668db","Node name for S&R":"GetImageSizeAndCount"},"widgets_values":[]},{"id":60,"type":"GetImageSizeAndCount","pos":[1279.781494140625,1060.245361328125],"size":[277.20001220703125,86],"flags":{},"order":22,"mode":0,"inputs":[{"name":"image","type":"IMAGE","link":139}],"outputs":[{"name":"image","type":"IMAGE","links":[151,152]},{"label":"704 width","name":"width","type":"INT","links":null},{"label":"544 height","name":"height","type":"INT","links":null},{"label":"1 count","name":"count","type":"INT","links":null}],"properties":{"cnr_id":"comfyui-kjnodes","ver":"8ecf5cd05e0a1012087b0da90eea9a13674668db","Node name for S&R":"GetImageSizeAndCount"},"widgets_values":[]},{"id":12,"type":"VAELoader","pos":[570.5363159179688,-282.70068359375],"size":[469.0488586425781,58],"flags":{},"order":4,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[153]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"VAELoader"},"widgets_values":["hunyuan_video_vae_bf16.safetensors"],"color":"#322","bgcolor":"#533"},{"id":66,"type":"SetNode","pos":[1083.503173828125,-358.4913330078125],"size":[210,60],"flags":{"collapsed":true},"order":15,"mode":0,"inputs":[{"name":"VAE","type":"VAE","link":153}],"outputs":[{"name":"*","type":"*","links":null}],"title":"Set_VAE","properties":{"previousName":"VAE"},"widgets_values":["VAE"],"color":"#322","bgcolor":"#533"},{"id":20,"type":"VAEEncode","pos":[1733.111083984375,633.30419921875],"size":[210,46],"flags":{},"order":24,"mode":0,"inputs":[{"name":"pixels","type":"IMAGE","link":156},{"name":"vae","type":"VAE","link":155}],"outputs":[{"name":"LATENT","type":"LATENT","links":[86]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"VAEEncode"},"widgets_values":[],"color":"#322","bgcolor":"#533"},{"id":68,"type":"GetNode","pos":[1729.60693359375,734.5352172851562],"size":[210,34],"flags":{"collapsed":true},"order":5,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[155]}],"title":"Get_VAE","properties":{},"widgets_values":["VAE"],"color":"#322","bgcolor":"#533"},{"id":62,"type":"VAEEncode","pos":[1612.563232421875,1048.6236572265625],"size":[210,46],"flags":{},"order":26,"mode":0,"inputs":[{"name":"pixels","type":"IMAGE","link":152},{"name":"vae","type":"VAE","link":158}],"outputs":[{"name":"LATENT","type":"LATENT","links":[147]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"VAEEncode"},"widgets_values":[],"color":"#322","bgcolor":"#533"},{"id":57,"type":"CLIPVisionEncode","pos":[1600.4202880859375,1181.36767578125],"size":[380.4000244140625,78],"flags":{},"order":25,"mode":0,"inputs":[{"name":"clip_vision","type":"CLIP_VISION","link":150},{"name":"image","type":"IMAGE","link":151}],"outputs":[{"name":"CLIP_VISION_OUTPUT","type":"CLIP_VISION_OUTPUT","links":[132]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.29","Node name for S&R":"CLIPVisionEncode"},"widgets_values":["center"],"color":"#233","bgcolor":"#355"},{"id":69,"type":"GetNode","pos":[1619.6104736328125,1137.854736328125],"size":[210,34],"flags":{"collapsed":true},"order":6,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[158]}],"title":"Get_VAE","properties":{},"widgets_values":["VAE"],"color":"#322","bgcolor":"#533"},{"id":65,"type":"GetNode","pos":[1604.746337890625,1306.3175048828125],"size":[210,34],"flags":{"collapsed":true},"order":7,"mode":0,"inputs":[],"outputs":[{"name":"CLIP_VISION","type":"CLIP_VISION","links":[150]}],"title":"Get_ClipVisionModle","properties":{},"widgets_values":["ClipVisionModle"],"color":"#233","bgcolor":"#355"},{"id":59,"type":"ImageResize+","pos":[908.9832763671875,1062.01123046875],"size":[315,218],"flags":{},"order":20,"mode":0,"inputs":[{"name":"image","type":"IMAGE","link":138},{"name":"width","type":"INT","widget":{"name":"width"},"link":136},{"name":"height","type":"INT","widget":{"name":"height"},"link":137}],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[139]},{"name":"width","type":"INT","links":null},{"name":"height","type":"INT","links":null}],"properties":{"aux_id":"kijai/ComfyUI_essentials","ver":"76e9d1e4399bd025ce8b12c290753d58f9f53e93","Node name for S&R":"ImageResize+"},"widgets_values":[512,512,"lanczos","stretch","always",0]},{"id":50,"type":"ImageResize+","pos":[907.2653198242188,593.743896484375],"size":[315,218],"flags":{},"order":19,"mode":0,"inputs":[{"name":"image","type":"IMAGE","link":122},{"name":"width","type":"INT","widget":{"name":"width"},"link":128},{"name":"height","type":"INT","widget":{"name":"height"},"link":127}],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[125]},{"name":"width","type":"INT","links":null},{"name":"height","type":"INT","links":null}],"properties":{"aux_id":"kijai/ComfyUI_essentials","ver":"76e9d1e4399bd025ce8b12c290753d58f9f53e93","Node name for S&R":"ImageResize+"},"widgets_values":[512,512,"lanczos","stretch","always",0]},{"id":58,"type":"LoadImage","pos":[190.07057189941406,1060.399169921875],"size":[315,314],"flags":{},"order":8,"mode":0,"inputs":[],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[138]},{"name":"MASK","type":"MASK","links":null}],"title":"Load Image: End","properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"LoadImage"},"widgets_values":["sd3stag.png","image"]},{"id":51,"type":"FramePackFindNearestBucket","pos":[550.0997314453125,887.411376953125],"size":[315,78],"flags":{},"order":16,"mode":0,"inputs":[{"name":"image","type":"IMAGE","link":126}],"outputs":[{"name":"width","type":"INT","links":[128,136]},{"name":"height","type":"INT","links":[127,137]}],"properties":{"aux_id":"kijai/ComfyUI-FramePackWrapper","ver":"4f9030a9f4c0bd67d86adf3d3dc07e37118c40bd","Node name for S&R":"FramePackFindNearestBucket"},"widgets_values":[512]},{"id":19,"type":"LoadImage","pos":[184.2612762451172,591.6886596679688],"size":[315,314],"flags":{},"order":9,"mode":0,"inputs":[],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[122,126]},{"name":"MASK","type":"MASK","links":null}],"title":"Load Image: Start","properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"LoadImage"},"widgets_values":["sd3stag.png","image"]},{"id":18,"type":"CLIPVisionLoader","pos":[33.149566650390625,23.595293045043945],"size":[388.87139892578125,58],"flags":{},"order":10,"mode":0,"inputs":[],"outputs":[{"name":"CLIP_VISION","type":"CLIP_VISION","links":[148]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"CLIPVisionLoader"},"widgets_values":["sigclip_vision_patch14_384.safetensors"],"color":"#2a363b","bgcolor":"#3f5159"},{"id":63,"type":"SetNode","pos":[247.1346435546875,-28.502397537231445],"size":[210,60],"flags":{"collapsed":true},"order":17,"mode":0,"inputs":[{"name":"CLIP_VISION","type":"CLIP_VISION","link":148}],"outputs":[{"name":"*","type":"*","links":null}],"title":"Set_ClipVisionModle","properties":{"previousName":"ClipVisionModle"},"widgets_values":["ClipVisionModle"],"color":"#233","bgcolor":"#355"},{"id":27,"type":"FramePackTorchCompileSettings","pos":[623.3660278320312,-140.94215393066406],"size":[531.5999755859375,202],"flags":{},"order":11,"mode":0,"inputs":[],"outputs":[{"name":"torch_compile_args","type":"FRAMEPACKCOMPILEARGS","links":[]}],"properties":{"aux_id":"lllyasviel/FramePack","ver":"0e5fe5d7ca13c76fb8e13708f4b92e7c7a34f20c","Node name for S&R":"FramePackTorchCompileSettings"},"widgets_values":["inductor",false,"default",false,64,true,true]},{"id":33,"type":"VAEDecodeTiled","pos":[2328.923828125,-22.08228874206543],"size":[315,150],"flags":{},"order":28,"mode":0,"inputs":[{"name":"samples","type":"LATENT","link":85},{"name":"vae","type":"VAE","link":154}],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[96]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"VAEDecodeTiled"},"widgets_values":[256,64,64,8],"color":"#322","bgcolor":"#533"},{"id":67,"type":"GetNode","pos":[2342.01806640625,-76.06847381591797],"size":[210,60],"flags":{"collapsed":true},"order":12,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[154]}],"title":"Get_VAE","properties":{},"widgets_values":["VAE"],"color":"#322","bgcolor":"#533"},{"id":23,"type":"VHS_VideoCombine","pos":[2726.849853515625,-29.90264129638672],"size":[908.428955078125,334],"flags":{},"order":30,"mode":0,"inputs":[{"name":"images","type":"IMAGE","link":97},{"name":"audio","shape":7,"type":"AUDIO","link":null},{"name":"meta_batch","shape":7,"type":"VHS_BatchManager","link":null},{"name":"vae","shape":7,"type":"VAE","link":null}],"outputs":[{"name":"Filenames","type":"VHS_FILENAMES","links":null}],"properties":{"cnr_id":"comfyui-videohelpersuite","ver":"0a75c7958fe320efcb052f1d9f8451fd20c730a8","Node name for S&R":"VHS_VideoCombine"},"widgets_values":{"frame_rate":30,"loop_count":0,"filename_prefix":"FramePack","format":"video/h264-mp4","pix_fmt":"yuv420p","crf":19,"save_metadata":true,"trim_to_audio":false,"pingpong":false,"save_output":true}},{"id":44,"type":"GetImageSizeAndCount","pos":[2501.023193359375,-178.70773315429688],"size":[277.20001220703125,86],"flags":{},"order":29,"mode":0,"inputs":[{"name":"image","type":"IMAGE","link":96}],"outputs":[{"name":"image","type":"IMAGE","links":[97]},{"label":"704 width","name":"width","type":"INT","links":null},{"label":"544 height","name":"height","type":"INT","links":null},{"label":"145 count","name":"count","type":"INT","links":null}],"properties":{"cnr_id":"comfyui-kjnodes","ver":"8ecf5cd05e0a1012087b0da90eea9a13674668db","Node name for S&R":"GetImageSizeAndCount"},"widgets_values":[]},{"id":47,"type":"CLIPTextEncode","pos":[715.3054809570312,127.73457336425781],"size":[400,200],"flags":{},"order":14,"mode":0,"inputs":[{"name":"clip","type":"CLIP","link":102}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[114,118]}],"properties":{"cnr_id":"comfy-core","ver":"0.3.28","Node name for S&R":"CLIPTextEncode"},"widgets_values":["majestig stag in a forest"],"color":"#232","bgcolor":"#353"},{"id":52,"type":"LoadFramePackModel","pos":[1253.046630859375,-82.57657623291016],"size":[480.7601013183594,174],"flags":{},"order":13,"mode":0,"inputs":[{"name":"compile_args","shape":7,"type":"FRAMEPACKCOMPILEARGS","link":null},{"name":"lora","shape":7,"type":"FPLORA","link":null}],"outputs":[{"name":"model","type":"FramePackMODEL","links":[129]}],"properties":{"aux_id":"kijai/ComfyUI-FramePackWrapper","ver":"49fe507eca8246cc9d08a8093892f40c1180e88f","Node name for S&R":"LoadFramePackModel"},"widgets_values":["FramePackI2V_HY_fp8_e4m3fn.safetensors","bf16","fp8_e4m3fn","offload_device","sdpa"]},{"id":39,"type":"FramePackSampler","pos":[2292.58837890625,194.90232849121094],"size":[365.07305908203125,814.6473388671875],"flags":{},"order":27,"mode":0,"inputs":[{"name":"model","type":"FramePackMODEL","link":129},{"name":"positive","type":"CONDITIONING","link":114},{"name":"negative","type":"CONDITIONING","link":108},{"name":"start_latent","type":"LATENT","link":86},{"name":"image_embeds","shape":7,"type":"CLIP_VISION_OUTPUT","link":141},{"name":"end_latent","shape":7,"type":"LATENT","link":147},{"name":"end_image_embeds","shape":7,"type":"CLIP_VISION_OUTPUT","link":132},{"name":"initial_samples","shape":7,"type":"LATENT","link":null}],"outputs":[{"name":"samples","type":"LATENT","links":[85]}],"properties":{"aux_id":"kijai/ComfyUI-FramePackWrapper","ver":"8e5ec6b7f3acf88255c5d93d062079f18b43aa2b","Node name for S&R":"FramePackSampler"},"widgets_values":[30,true,0.15,1,10,0,47,"fixed",3,5,6,"unipc_bh1","weighted_average",0.5,1]}],"links":[[85,39,0,33,0,"LATENT"],[86,20,0,39,3,"LATENT"],[96,33,0,44,0,"IMAGE"],[97,44,0,23,0,"IMAGE"],[102,13,0,47,0,"CLIP"],[108,15,0,39,2,"CONDITIONING"],[114,47,0,39,1,"CONDITIONING"],[116,48,0,17,1,"IMAGE"],[118,47,0,15,0,"CONDITIONING"],[122,19,0,50,0,"IMAGE"],[125,50,0,48,0,"IMAGE"],[126,19,0,51,0,"IMAGE"],[127,51,1,50,2,"INT"],[128,51,0,50,1,"INT"],[129,52,0,39,0,"FramePackMODEL"],[132,57,0,39,6,"CLIP_VISION_OUTPUT"],[136,51,0,59,1,"INT"],[137,51,1,59,2,"INT"],[138,58,0,59,0,"IMAGE"],[139,59,0,60,0,"IMAGE"],[141,17,0,39,4,"CLIP_VISION_OUTPUT"],[147,62,0,39,5,"LATENT"],[148,18,0,63,0,"*"],[149,64,0,17,0,"CLIP_VISION"],[150,65,0,57,0,"CLIP_VISION"],[151,60,0,57,1,"IMAGE"],[152,60,0,62,0,"IMAGE"],[153,12,0,66,0,"*"],[154,67,0,33,1,"VAE"],[155,68,0,20,1,"VAE"],[156,48,0,20,0,"IMAGE"],[158,69,0,62,1,"VAE"]],"groups":[{"id":1,"title":"End Image","bounding":[12.77297592163086,999.1203002929688,2038.674560546875,412.9618225097656],"color":"#3f789e","font_size":24,"flags":{}},{"id":2,"title":"Start Image","bounding":[11.781991958618164,531.3884887695312,2032.7288818359375,442.6904602050781],"color":"#3f789e","font_size":24,"flags":{}}],"config":{},"extra":{"ds":{"scale":0.6115909044841659,"offset":[21.57747102795121,375.7674957811538]},"frontendVersion":"1.18.3","VHS_latentpreview":true,"VHS_latentpreviewrate":0,"VHS_MetadataImage":true,"VHS_KeepIntermediate":true},"version":0.4}''')

with open(f"{WF_DIR}/framepack_i2v.json", "w", encoding="utf-8") as f:
    json.dump(WORKFLOW, f, ensure_ascii=False)
print("✅ ワークフローを配置しました: framepack_i2v.json")

## Step 4: ComfyUI の起動
cloudflared でトンネルを開きます。表示された `trycloudflare.com` の URL を開いてください。

In [ ]:
#@title Step 4: ComfyUI を起動（cloudflared）

!wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess, threading, time, socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break
    print("\nComfyUI の起動完了。cloudflared でトンネルを開きます...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("🌐 ComfyUI アクセス URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
# VRAM が厳しい場合は --lowvram を追加してください
!python main.py --dont-print-server --listen --enable-cors-header '*'

## Step 5: スプライトシートの作成（ゲーム用キャラクター向け）

生成した動画から、等間隔にフレームを抜き出して **背景透過の1枚のPNG** にまとめます。

> ⚠️ **Step 4 の ComfyUI を先に停止してください**（セルの ■ ボタン）。起動中は次のセルを実行できません。
> 動画は Google Drive の `ComfyUI_Output` に保存済みなので、停止しても消えません。

- 既定: 20コマ / 10列（=10列×2行）/ 高さ256px / 背景透過
- 同じフォルダに `*_sprite.png` と、ゲームエンジン用のメタ情報 `*_sprite.json` を出力します

In [ ]:
#@title Step 5: スプライトシートを作成（透過PNG）

import os, glob, json, math
import numpy as np
import cv2
from PIL import Image

OUTPUT_DIR   = "/content/drive/MyDrive/ComfyUI_Output"
VIDEO_PATH   = ""      #@param {type:"string"} 空欄なら出力フォルダの最新の mp4 を使用
N_FRAMES     = 20      #@param {type:"integer"} シートに並べるフレーム数（動画から等間隔に抽出）
COLUMNS      = 10      #@param {type:"integer"} 列数（行数は自動計算）
FRAME_HEIGHT = 256     #@param {type:"integer"} 1コマの高さ(px)。0なら元の解像度のまま
REMOVE_BG    = True    #@param {type:"boolean"} 背景を透過にする（rembg）
BG_MODEL     = "isnet-anime"  #@param ["isnet-anime", "u2net", "isnet-general-use"] キャラ絵ならisnet-anime推奨
DROP_LAST    = False   #@param {type:"boolean"} 最後のコマを除く（ループ素材で先頭と重なる場合）

# ── 入力動画 ──
video = VIDEO_PATH
if not video:
    cands = glob.glob(f"{OUTPUT_DIR}/**/*.mp4", recursive=True)
    assert cands, f"{OUTPUT_DIR} に mp4 が見つかりません。先に動画を生成してください"
    video = max(cands, key=os.path.getmtime)
print("🎞️ 入力:", video)

# ── 全フレーム読み込み ──
cap = cv2.VideoCapture(video)
frames = []
while True:
    ok, bgr = cap.read()
    if not ok:
        break
    frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
cap.release()
total = len(frames)
assert total > 0, "動画からフレームを読み込めませんでした"
print(f"   総フレーム数: {total}")

# ── 等間隔に抽出 ──
n = min(N_FRAMES, total)
idx = np.linspace(0, total - 1, n + (1 if DROP_LAST else 0)).round().astype(int)
if DROP_LAST:
    idx = idx[:-1]
picked = [Image.fromarray(frames[i]) for i in idx]
print(f"   抽出: {len(picked)} コマ")

# ── リサイズ ──
if FRAME_HEIGHT > 0:
    h = FRAME_HEIGHT
    w = round(picked[0].width * h / picked[0].height)
    picked = [im.resize((w, h), Image.LANCZOS) for im in picked]

# ── 背景透過 ──
if REMOVE_BG:
    !pip install -q rembg onnxruntime
    from rembg import remove, new_session
    session = new_session(BG_MODEL)
    picked = [remove(im, session=session) for im in picked]
else:
    picked = [im.convert("RGBA") for im in picked]

# ── シートに並べる ──
fw, fh = picked[0].size
cols = max(1, min(COLUMNS, len(picked)))
rows = math.ceil(len(picked) / cols)
sheet = Image.new("RGBA", (fw * cols, fh * rows), (0, 0, 0, 0))
for i, im in enumerate(picked):
    sheet.paste(im, ((i % cols) * fw, (i // cols) * fh))

base = os.path.splitext(os.path.basename(video))[0]
png_path  = f"{OUTPUT_DIR}/{base}_sprite.png"
json_path = f"{OUTPUT_DIR}/{base}_sprite.json"
sheet.save(png_path)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({"frame_width": fw, "frame_height": fh, "columns": cols,
               "rows": rows, "frame_count": len(picked),
               "source_frame_indices": [int(i) for i in idx]}, f, indent=2)

print(f"\n✅ 保存しました: {png_path}  ({sheet.width}x{sheet.height}, {cols}列 x {rows}行, 1コマ {fw}x{fh})")
print(f"   メタ情報: {json_path}")
display(sheet.convert("RGBA"))